In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load data
df = pd.read_csv("/Users/kushgirap/Desktop/bbl-2026-analysis/data/bbl_2026_final_coded - Sheet3.csv")

# Quick check
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst row:\n{df.head(1)}")

Shape: (239, 28)

Columns: ['Innings', 'over ', 'ball', 'striker', 'non striker', 'bowler', 'type of bowler', 'Bowling end', 'Runs batter', 'Extras', 'Wide', 'No ball', 'Leg bye', 'Bye', 'Wicket', 'Dismissal Type', 'caught at', 'fielder', 'Line ', 'Length', 'Shot played', 'Intended direction', 'actual direction', 'Reason for mismatch', 'Middled', 'Edged', 'Beaten', 'Ball speed (Kmph)']

First row:
   Innings  over   ball         striker   non striker       bowler  \
0        1      0     1  Matthew Gilkes  David Warner  Sean Abbott   

      type of bowler Bowling end  Runs batter  Extras  ...        Line   \
0  Right Medium Fast        Over            0     1.0  ...  Outside Off   

        Length   Shot played Intended direction actual direction  \
0  Good Length  Square drive              Point           Keeper   

  Reason for mismatch Middled Edged Beaten Ball speed (Kmph)  
0                Wide      No    No     No               NaN  

[1 rows x 28 columns]


In [35]:
# ── Clean column names ────────────────────────────────────────────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Cleaned columns:")
print(df.columns.tolist())

# ── Check data types ──────────────────────────────────────────────────────────
print("\nData types:")
print(df.dtypes)

# ── Check unique values in key columns ───────────────────────────────────────
for col in ["innings", "type_of_bowler", "line", "length", "shot_played", "wide", "wicket"]:
    print(f"\n{col}: {df[col].unique()}")

Cleaned columns:
['innings', 'over', 'ball', 'striker', 'non_striker', 'bowler', 'type_of_bowler', 'bowling_end', 'runs_batter', 'extras', 'wide', 'no_ball', 'leg_bye', 'bye', 'wicket', 'dismissal_type', 'caught_at', 'fielder', 'line', 'length', 'shot_played', 'intended_direction', 'actual_direction', 'reason_for_mismatch', 'middled', 'edged', 'beaten', 'ball_speed', 'is_wicket', 'is_wide', 'is_no_ball', 'is_bye', 'is_leg_bye', 'is_beaten', 'is_middled', 'is_edged', 'is_dot', 'is_boundary', 'is_legal', 'is_four', 'is_six', 'direction_zone', 'prev_dot1', 'prev_dot2', 'pressure_sequence']

Data types:
innings                  int64
over                     int64
ball                     int64
striker                 object
non_striker             object
bowler                  object
type_of_bowler          object
bowling_end             object
runs_batter              int64
extras                 float64
wide                    object
no_ball                 object
leg_bye              

In [36]:
# ── Rename ball speed column ──────────────────────────────────────────────────
df = df.rename(columns={"ball_speed_(kmph)": "ball_speed"})

# ── Fill NaN values ───────────────────────────────────────────────────────────
df["extras"]       = df["extras"].fillna(0)
df["ball_speed"]   = pd.to_numeric(df["ball_speed"], errors="coerce")

# ── Boolean helper columns ────────────────────────────────────────────────────
df["is_wicket"]    = df["wicket"]  == "Yes"
df["is_wide"]      = df["wide"]    == "Yes"
df["is_no_ball"]   = df["no_ball"] == "Yes"
df["is_bye"]       = df["bye"]     == "Yes"
df["is_leg_bye"]   = df["leg_bye"] == "Yes"
df["is_beaten"]    = df["beaten"]  == "Yes"
df["is_middled"]   = df["middled"]  == "Yes"
df["is_edged"]     = df["edged"]    == "Yes"
df["is_dot"]       = (df["runs_batter"] == 0) & (df["is_wide"] == False) & (df["extras"] == 0)
df["is_boundary"]  = df["runs_batter"].isin([4, 6])
df["is_legal"]     = (df["is_wide"] == False) 


# ── Quick sanity check ────────────────────────────────────────────────────────
print(f"Total deliveries:  {len(df)}")
print(f"Legal deliveries:  {df['is_legal'].sum()}")
print(f"Wides:             {df['is_wide'].sum()}")
print(f"Wickets:           {df['is_wicket'].sum()}")
print(f"Boundaries:        {df['is_boundary'].sum()}")
print(f"Innings:           {df['innings'].unique()}")
print(f"Bowler types:      {df['type_of_bowler'].unique()}")

Total deliveries:  239
Legal deliveries:  227
Wides:             12
Wickets:           11
Boundaries:        50
Innings:           [1 2]
Bowler types:      ['Right Medium Fast' 'Left Fast' 'Right Off' 'Left Off' 'Left Medium Fast'
 'Right Fast' 'Right Leg']


In [37]:
df["is_four"] = (df["is_boundary"] == True) & (df["runs_batter"] == 4)
df["is_six"]  = (df["is_boundary"] == True) & (df["runs_batter"] == 6)

In [39]:
batting_vs_bowler = df[df["is_legal"]].groupby(
    ["striker", "bowler", "type_of_bowler"]
).agg(
    balls      = ("ball", "count"),
    runs       = ("runs_batter", "sum"),
    dot_balls  = ("is_dot", "sum"),
    wickets    = ("is_wicket", "sum"),
    boundaries = ("is_boundary", "sum"),
    fours      = ("is_four", "sum"),
    sixes      = ("is_six", "sum")
).reset_index()

batting_vs_bowler["dot_pct"]     = (batting_vs_bowler["dot_balls"] / batting_vs_bowler["balls"] * 100).round(1)
batting_vs_bowler["strike_rate"] = (batting_vs_bowler["runs"] / batting_vs_bowler["balls"] * 100).round(1)

display(batting_vs_bowler[batting_vs_bowler["balls"] >= 6].sort_values(
    ["striker", "strike_rate"], ascending=[True, False]
))

,striker,bowler,type_of_bowler,balls,runs,dot_balls,wickets,boundaries,fours,sixes,dot_pct,strike_rate
4,Babar Azam,Tanveer Sangha,Right Leg,10,16,1,0,2,2,0,10.0,160.0
1,Babar Azam,Chris Green,Right Off,15,12,8,0,2,2,0,53.3,80.0
9,David Warner,Jack Edwards,Right Medium Fast,14,33,2,0,5,4,1,14.3,235.7
13,David Warner,Sean Abbott,Right Medium Fast,20,44,5,0,8,6,2,25.0,220.0
11,David Warner,Mitchell Starc,Left Fast,14,15,6,0,1,0,1,42.9,107.1
8,David Warner,Ben Manenti,Right Off,6,6,1,0,0,0,0,16.7,100.0
12,David Warner,Sam Curran,Left Medium Fast,8,5,3,0,0,0,0,37.5,62.5
26,Nic Maddinson,Ben Manenti,Right Off,6,8,2,0,1,1,0,33.3,133.3
44,Steve Smith,Ryan Hadley,Right Fast,9,32,1,0,5,1,4,11.1,355.6
43,Steve Smith,Nathan McAndrew,Right Medium Fast,8,28,1,0,5,2,3,12.5,350.0


In [44]:
wicket_df = df[df["is_wicket"] == True]

wicket_breakdown = wicket_df.groupby(["line", "length", "dismissal_type"]).size().reset_index(name="wickets")
display(wicket_breakdown.sort_values("wickets", ascending=False))

,line,length,dismissal_type,wickets
3,Middle,Good Length,Caught,3
4,Outside Off,Bouncer,Caught,2
0,Leg,Bouncer,Caught,1
1,Leg,Good Length,Caught and Bowled,1
2,Middle,Full,LBW,1
5,Outside Off,Good Length,Caught,1
6,Outside Off,Short of length,Bowled,1
7,Outside Off,Short of length,Stumped,1


In [6]:
boundary_df = df[df["is_boundary"] == True]

boundary_direction = boundary_df.groupby("actual_direction").size().reset_index(name="boundaries")
boundary_direction["pct"] = (boundary_direction["boundaries"] / boundary_direction["boundaries"].sum() * 100).round(1)
display(boundary_direction.sort_values("boundaries", ascending=False))

,actual_direction,boundaries,pct
1,Deep Backward Square Leg,6,12.0
6,Deep Mid Wicket,6,12.0
7,Deep Square Leg,6,12.0
5,Deep Forward Square Leg,5,10.0
11,Straight Hit,5,10.0
0,Deep Backward Point,4,8.0
12,Sweeper Cover,4,8.0
2,Deep Cover,3,6.0
3,Deep Fine Leg,3,6.0
4,Deep Forward Point,2,4.0


In [46]:
line_length = df[df["is_legal"]].groupby(["line", "length"]).agg(
    balls      = ("ball", "count"),
    runs       = ("runs_batter", "sum"),
    wickets    = ("is_wicket", "sum"),
    dots       = ("is_dot", "sum"),
    boundaries = ("is_boundary", "sum"),
    beat_bat   = ("beaten", lambda x: (x == "Yes").sum())
).reset_index()

line_length["economy"]  = (line_length["runs"] / (line_length["balls"] / 6)).round(2)
line_length["dot_pct"]  = (line_length["dots"] / line_length["balls"] * 100).round(1)
display(line_length[line_length["balls"] >= 10].sort_values("dot_pct", ascending=False))

,line,length,balls,runs,wickets,dots,boundaries,beat_bat,economy,dot_pct
8,Middle,Good Length,22,25,3,10,3,0,6.82,45.5
24,Outside Off,Good Length,41,63,1,17,9,9,9.22,41.5
18,Outside Leg,Good Length,16,20,0,6,4,3,7.50,37.5
12,Off,Full,10,13,0,3,2,0,7.80,30.0
25,Outside Off,Short of length,18,33,2,5,6,3,11.00,27.8
13,Off,Good Length,16,27,0,4,4,1,10.12,25.0
9,Middle,Short of length,13,27,0,3,3,1,12.46,23.1
7,Middle,Full,17,27,1,3,2,1,9.53,17.6


In [47]:
bowler_threat = df[df["is_legal"]].groupby("bowler").agg(
    balls      = ("ball", "count"),
    runs       = ("runs_batter", "sum"),
    wickets    = ("is_wicket", "sum"),
    beat_bat   = ("beaten", lambda x: (x == "Yes").sum()),
    edges      = ("edged", lambda x: (x == "Yes").sum()),
    bowler_type = ("type_of_bowler", "first")
).reset_index()

bowler_threat["economy"]       = (bowler_threat["runs"] / (bowler_threat["balls"] / 6)).round(2)
bowler_threat["beat_bat_pct"]  = (bowler_threat["beat_bat"] / bowler_threat["balls"] * 100).round(1)
display(bowler_threat.sort_values("economy", ascending=True))

,bowler,balls,runs,wickets,beat_bat,edges,bowler_type,economy,beat_bat_pct
2,Chris Green,24,18,1,2,2,Right Off,4.50,8.3
1,Ben Manenti,18,17,1,4,1,Right Off,5.67,22.2
9,Sam Curran,24,25,3,2,1,Left Medium Fast,6.25,8.3
5,Mitchell Starc,24,29,1,3,3,Left Fast,7.25,12.5
0,Aidan O'Connor,12,16,1,2,1,Right Medium Fast,8.00,16.7
11,Tanveer Sangha,24,33,1,2,0,Right Leg,8.25,8.3
4,Joel Davis,6,9,0,0,0,Left Off,9.00,0.0
10,Sean Abbott,26,46,0,7,1,Right Medium Fast,10.62,26.9
12,Wes Agar,12,24,0,2,1,Right Medium Fast,12.00,16.7
3,Jack Edwards,24,52,1,4,4,Right Medium Fast,13.00,16.7


In [10]:
mismatch_df = df[df["reason_for_mismatch"].notna() & (df["reason_for_mismatch"] != "")]

mismatch_breakdown = mismatch_df.groupby(["line", "length", "shot_played", "reason_for_mismatch"]).size().reset_index(name="count")
display(mismatch_breakdown.sort_values("count", ascending=False).head(10))

,line,length,shot_played,reason_for_mismatch,count
50,Outside Off,Good Length,Square drive,Beaten,3
1,Leg,Bouncer,Hook,Wide,2
25,Off,Good Length,Forward Defence,Outside Edge,2
40,Outside Off,Bouncer,Pull Shot,Outside Edge,2
47,Outside Off,Good Length,Slog,Beaten,2
36,Outside Leg,Yorker,Flick,Beaten,2
57,Outside Off,Short of length,Square Cut,Outside Edge,2
44,Outside Off,Good Length,Lofted Drive,Beaten,2
35,Outside Leg,Short of length,Pull Shot,Wide,1
34,Outside Leg,Good Length,Step out,Beaten,1


In [27]:
beaten_df = df[df["beaten"] == "Yes"]

shot_beaten = beaten_df.groupby(
    [ "bowler", "line", "length"]
).size().reset_index(name="times_beaten")

display(shot_beaten.sort_values("times_beaten", ascending=False).head(15))

,bowler,line,length,times_beaten
8,Jack Edwards,Outside Off,Good Length,3
4,Ben Manenti,Outside Off,Good Length,2
18,Sean Abbott,Outside Leg,Yorker,2
0,Aidan O'Connor,Leg,Short of length,1
14,Sam Curran,Outside Leg,Good Length,1
23,Wes Agar,Off,Bouncer,1
22,Tanveer Sangha,Outside Off,Short of length,1
21,Tanveer Sangha,Outside Off,Good Length,1
20,Sean Abbott,Outside Off,Yorker,1
19,Sean Abbott,Outside Off,Good Length,1


In [28]:
direction_map = {
    "Fine Leg"              : "Leg Side",
    "Deep Fine Leg"         : "Leg Side",
    "Square Leg"            : "Leg Side",
    "Deep Square Leg"       : "Leg Side",
    "Deep Backward Square Leg": "Leg Side",
    "Forward Square Leg"    : "Leg Side",
    "Deep Forward Square Leg": "Leg Side",
    "Mid Wicket"            : "Leg Side",
    "Deep Mid Wicket"       : "Leg Side",
    "Mid On"                : "Straight",
    "Long On"               : "Straight",
    "Straight Hit"          : "Straight",
    "Long Off"              : "Straight",
    "Mid Off"               : "Off Side",
    "Extra Cover"           : "Off Side",
    "Deep Cover"            : "Off Side",
    "Cover"                 : "Off Side",
    "Sweeper Cover"         : "Off Side",
    "Point"                 : "Off Side",
    "Deep Forward Point"    : "Off Side",
    "Backward Point"        : "Off Side",
    "Deep Backward Point"   : "Off Side",
    "Gully"                 : "Off Side",
    "Third Man"             : "Off Side",
    "Fine Third Man"        : "Off Side",
    "Deep Third Man"        : "Off Side",
}

df["direction_zone"] = df["actual_direction"].map(direction_map)

zone_analysis = df[df["is_boundary"]].groupby("direction_zone").agg(
    boundaries = ("is_boundary", "sum"),
    fours      = ("is_four", "sum"),
    sixes      = ("is_six", "sum")
).reset_index()
zone_analysis["pct"] = (zone_analysis["boundaries"] / zone_analysis["boundaries"].sum() * 100).round(1)
print(zone_analysis.sort_values("boundaries", ascending=False))

  direction_zone  boundaries  fours  sixes   pct
0       Leg Side          26     13     13  52.0
1       Off Side          17     13      4  34.0
2       Straight           7      5      2  14.0


In [29]:
df["prev_dot1"] = df.groupby(["innings", "bowler"])["is_dot"].shift(1)
df["prev_dot2"] = df.groupby(["innings", "bowler"])["is_dot"].shift(2)
df["pressure_sequence"] = (df["is_dot"] == True) & (df["prev_dot1"] == True) & (df["prev_dot2"] == True)

pressure = df.groupby("bowler")["pressure_sequence"].sum().reset_index()
pressure.columns = ["bowler", "pressure_sequences"]
print(pressure.sort_values("pressure_sequences", ascending=False))

             bowler  pressure_sequences
2      Chris Green                    2
5    Mitchell Starc                   1
11   Tanveer Sangha                   1
0    Aidan O'Connor                   0
1       Ben Manenti                   0
3      Jack Edwards                   0
4        Joel Davis                   0
6   Nathan McAndrew                   0
7     Nic Maddinson                   0
8       Ryan Hadley                   0
9        Sam Curran                   0
10      Sean Abbott                   0
12         Wes Agar                   0


In [30]:
bowler_ratio = df[df["is_legal"]].groupby("bowler").agg(
    wickets    = ("is_wicket", "sum"),
    boundaries = ("is_boundary", "sum")
).reset_index()
bowler_ratio["wicket_to_boundary"] = (bowler_ratio["wickets"] / (bowler_ratio["boundaries"] + 1)).round(2)
print(bowler_ratio.sort_values("wicket_to_boundary", ascending=False))

             bowler  wickets  boundaries  wicket_to_boundary
9        Sam Curran        3           3                0.75
1       Ben Manenti        1           1                0.50
0    Aidan O'Connor        1           2                0.33
2      Chris Green         1           2                0.33
6   Nathan McAndrew        2           6                0.29
5    Mitchell Starc        1           3                0.25
11   Tanveer Sangha        1           4                0.20
3      Jack Edwards        1           8                0.11
4        Joel Davis        0           1                0.00
7     Nic Maddinson        0           2                0.00
8       Ryan Hadley        0           7                0.00
10      Sean Abbott        0           8                0.00
12         Wes Agar        0           3                0.00
